# Examen práctico: grupo 1, el MDP

Desarrollamos los entregables 1.1, 1.2, 1.3 y 1.4 sobre la representación del estado y la función de transición. Integramos los resultados disponibles de los siete grupos para responder las cuatro preguntas y elaborar la reflexión grupal.

Construimos ejemplos ilustrativos para analizar los supuestos del modelo y los distinguimos de las métricas de producción proporcionadas. En la integración explicamos cómo las decisiones de cada componente afectan el funcionamiento conjunto y señalamos las adaptaciones pendientes.


## Modelo que analizamos

Conservamos la transición original y representamos el estado como $(I_t,E_t,L_t)$: inventario, días hasta el vencimiento y categoría de demanda promedio de siete días. Interpretamos la acción como unidades pedidas y observamos una reposición inmediata, con inventario final limitado entre 0 y 100. En esta etapa analizamos sus supuestos sin implementar todavía un MDP corregido.


In [1]:
def transition(state, action):
    """Calcula el estado siguiente de forma determinista; la acción son unidades y la categoría de demanda permanece fija."""
    inventory, days_to_expiry, demand_level = state
    demand_map = {'bajo': 5, 'medio': 15, 'alto': 25, 'crítico': 40}
    daily_demand = demand_map[demand_level]
    new_inventory = min(100, max(0, inventory + action - daily_demand))
    new_days = max(1, days_to_expiry - 1)
    new_demand = demand_level
    return (new_inventory, new_days, new_demand)


resultados_produccion = {
    'stockouts_por_semana': 23,
    'productos_vencidos_por_semana': 41,
    'costo_almacenamiento_semanal': 8400,
    'costo_objetivo_semanal': 3200,
    'satisfaccion_cliente': 0.61,
}


## Entregable 1.1: propiedad de Markov

Usamos $H_t=(S_0,A_0,\ldots,S_t)$ para representar la historia observable. Exigimos que, al conocer el estado actual y la acción, la historia no aporte información adicional para predecir el siguiente estado:

$$P(S_{t+1}=s'\mid H_t,A_t=a)=P(S_{t+1}=s'\mid S_t,A_t=a).$$

**En el simulador.** Observamos que `transition` depende únicamente del estado actual y la acción. Si representamos esa función mediante $f$, obtenemos $P_{sim}(s'\mid s,a)=\mathbf{1}\{s'=f(s,a)\}$. Concluimos que la transición implementada sí cumple Markov; no confundimos una demanda poco realista con un incumplimiento de esta propiedad. Limitamos la conclusión a la transición disponible, porque no contamos con la implementación completa del entorno.

**En la farmacia real.** Identificamos información ausente que podría modificar la predicción del estado siguiente:

* No observamos pedidos pendientes, cantidades ni fechas de llegada; podemos tener el mismo inventario actual y distintas entregas al día siguiente.
* No conservamos la secuencia de demandas de los últimos siete días, su orden ni la media exacta al representar la demanda mediante una categoría.
* No incorporamos calendario, tendencias ni eventos que puedan alterar la distribución de demanda futura.
* No distinguimos cantidades y vencimientos por lote, por lo que no podemos determinar cuántas unidades dejarían de ser utilizables mañana.

Planteamos estas ausencias como mecanismos posibles del dominio, sin afirmar que hayan ocurrido en los registros de producción. Tampoco consideramos la criticidad clínica, por sí sola, una prueba de que la transición incumpla Markov.


### Contraejemplo con dos historias de pedidos

Construimos dos historias con el mismo estado observado $s=(20,14,\text{medio})$ y la misma acción $a=0$. En la primera suponemos una entrega pendiente de 30 unidades antes de las ventas de mañana; en la segunda suponemos que no existe esa entrega. Fijamos la demanda en 15 unidades para aislar el efecto de la información omitida.


In [2]:
estado_compartido = (20, 14, 'medio')
accion_compartida = 0
inventario_con_pedido_previo = min(100, max(0, 20 + 30 - 15))
inventario_sin_pedido_previo = min(100, max(0, 20 + 0 - 15))
print('Ejemplo hipotético: mismo estado y acción, distintas historias')
print(f'Historia con entrega pendiente: inventario siguiente = {inventario_con_pedido_previo}')
print(f'Historia sin entrega pendiente: inventario siguiente = {inventario_sin_pedido_previo}')
print(f'Transición original: {transition(estado_compartido, accion_compartida)}')
assert inventario_con_pedido_previo == 35
assert inventario_sin_pedido_previo == 5


Ejemplo hipotético: mismo estado y acción, distintas historias
Historia con entrega pendiente: inventario siguiente = 35
Historia sin entrega pendiente: inventario siguiente = 5
Transición original: (5, 13, 'medio')


Obtenemos 35 unidades con la entrega pendiente y 5 sin ella. Expresamos la diferencia como $P(I_{t+1}=35\mid H_t^{(1)},a=0)=1$ y $P(I_{t+1}=35\mid H_t^{(2)},a=0)=0$, aunque terminamos ambas historias en el mismo estado observado. Si admitimos ese funcionamiento real, no podemos representar ambas probabilidades mediante una transición independiente de la historia. Demostramos insuficiencia bajo ese supuesto, sin afirmar que existan pedidos pendientes en los datos entregados.

### Pérdida de información de la demanda

Interpretamos el promedio de siete días como una media móvil y escribimos su actualización como $m_{t+1}=m_t+(d_{t+1}-d_{t-6})/7$. No contamos en la tupla con la demanda más antigua $d_{t-6}$ que debemos retirar. Para mostrar la pérdida de información, construimos dos ventanas ordenadas de más antigua a más reciente, con igual media actual e igual demanda de mañana.


In [3]:
ventana_a = [25, 10, 10, 15, 15, 15, 15]
ventana_b = [10, 10, 15, 15, 15, 15, 25]
demanda_manana = 15
media_actual_a = sum(ventana_a) / 7
media_actual_b = sum(ventana_b) / 7
media_siguiente_a = (sum(ventana_a[1:]) + demanda_manana) / 7
media_siguiente_b = (sum(ventana_b[1:]) + demanda_manana) / 7
print(f'Medias actuales: {media_actual_a:.2f} y {media_actual_b:.2f}')
print(f'Medias siguientes: {media_siguiente_a:.2f} y {media_siguiente_b:.2f}')
assert media_actual_a == media_actual_b == 15
assert media_siguiente_a != media_siguiente_b


Medias actuales: 15.00 y 15.00
Medias siguientes: 13.57 y 15.71


Obtenemos medias siguientes de 13.57 y 15.71 a partir de una misma media actual de 15. Mostramos así que no podemos reconstruir la media siguiente con la información conservada. Como no disponemos de umbrales de clasificación, no afirmamos que ambas medias deban corresponder a categorías distintas.

**Consecuencia sobre el aprendizaje.** Llamamos $Z_t$ a la información omitida. Si suponemos que un estado ampliado $(S_t,Z_t)$ es suficiente, expresamos la transición observada como:

$$P(s'\mid H_t,a)=\sum_z P(s'\mid s,z,a)P(z\mid H_t,a).$$

Si cambiamos las probabilidades de la información oculta según la historia y esa información modifica las transiciones, obtenemos futuros distintos para un mismo par $(s,a)$. Al utilizar una única entrada $Q(s,a)$, agrupamos situaciones que pueden requerir decisiones diferentes; con una política que solo consulta $s$, no podemos distinguirlas.

Por ello no podemos aplicar directamente las garantías de un MDP estacionario a una representación real cuya suficiencia no hemos demostrado. No concluimos que el aprendizaje deba divergir: podemos obtener un compromiso o aprender correctamente el simulador y fallar al transferir la política a producción.


## Entregable 1.2: análisis de la función de transición

Extraemos las siguientes ecuaciones de la transición:

$$I_{t+1}=\min(100,\max(0,I_t+a_t-d(L_t))).$$

$$E_{t+1}=\max(1,E_t-1),\qquad L_{t+1}=L_t.$$

Usamos las demandas definidas en el modelo: $d(\text{bajo})=5$, $d(\text{medio})=15$, $d(\text{alto})=25$ y $d(\text{crítico})=40$. Por inducción obtenemos $L_t=L_0$ para cualquier secuencia de acciones. También observamos que, dentro de cada categoría, mantenemos una demanda diaria determinista con varianza condicional cero.

Concluimos que con este supuesto no representamos fluctuaciones diarias, estacionalidad, emergencias, tendencias ni caídas de rotación. Tampoco actualizamos la media de siete días. Identificamos cuatro subconjuntos sin transiciones entre categorías: si comenzamos en `bajo`, nunca llegamos a `crítico` dentro de esa trayectoria. Como no conocemos la inicialización del entorno, no descartamos que otros episodios comiencen en `crítico`.


In [4]:
estado = (60, 14, 'bajo')
trayectoria_original = [estado]
for _ in range(4):
    estado = transition(estado, 0)
    trayectoria_original.append(estado)

print('Transición original con acción de cero unidades')
print('Día | Inventario | Días al vencimiento | Demanda')
for dia, (inventario, vencimiento, nivel) in enumerate(trayectoria_original):
    print(f'{dia} | {inventario} | {vencimiento} | {nivel}')
assert all(s[2] == 'bajo' for s in trayectoria_original)


Transición original con acción de cero unidades
Día | Inventario | Días al vencimiento | Demanda
0 | 60 | 14 | bajo
1 | 55 | 13 | bajo
2 | 50 | 12 | bajo
3 | 45 | 11 | bajo
4 | 40 | 10 | bajo


### Sensibilidad a demandas que el modelo no contempla

Calculamos balances para secuencias ilustrativas y conservamos el recorte del inventario original. Mantenemos idénticos el inventario inicial y las acciones dentro de cada comparación, variamos únicamente la demanda y registramos las unidades no atendidas. Utilizamos esta función como herramienta de diagnóstico, sin definir todavía un nuevo estado ni una transición corregida para el agente.


In [5]:
def balance_diagnostico(inventario_inicial, acciones, demandas):
    """Compara demandas hipotéticas en el balance del MDP y registra unidades no atendidas que el recorte a cero oculta."""
    inventario = inventario_inicial
    filas = []
    for dia, (accion, demanda) in enumerate(zip(acciones, demandas), start=1):
        disponible = inventario + accion
        faltante = max(0, demanda - disponible)
        inventario = min(100, max(0, disponible - demanda))
        filas.append((dia, demanda, inventario, faltante))
    return filas


casos = [
    ('Demanda fija', 60, [0] * 4, [5, 5, 5, 5]),
    ('Pico hipotético', 60, [0] * 4, [5, 40, 40, 5]),
    ('Rotación fija', 50, [10] * 4, [5, 5, 5, 5]),
    ('Caída hipotética', 50, [10] * 4, [0, 0, 0, 0]),
]
balances = {}
print('Escenario | Demanda por día | Inventarios al cierre | Unidades no atendidas')
for nombre, inicial, acciones, demandas in casos:
    filas = balance_diagnostico(inicial, acciones, demandas)
    balances[nombre] = filas
    cierres = [fila[2] for fila in filas]
    no_atendidas = sum(fila[3] for fila in filas)
    print(f'{nombre} | {demandas} | {cierres} | {no_atendidas}')
assert [fila[2] for fila in balances['Demanda fija']] == [s[0] for s in trayectoria_original[1:]]
assert sum(fila[3] for fila in balances['Pico hipotético']) == 30
assert balances['Rotación fija'][-1][2] == 70
assert balances['Caída hipotética'][-1][2] == 90


Escenario | Demanda por día | Inventarios al cierre | Unidades no atendidas
Demanda fija | [5, 5, 5, 5] | [55, 50, 45, 40] | 0
Pico hipotético | [5, 40, 40, 5] | [55, 15, 0, 0] | 30
Rotación fija | [5, 5, 5, 5] | [55, 60, 65, 70] | 0
Caída hipotética | [0, 0, 0, 0] | [60, 70, 80, 90] | 0


Observamos que con demanda fija terminamos con 40 unidades y ningún faltante, mientras que con el pico terminamos sin inventario y acumulamos 30 unidades no atendidas. En la segunda comparación obtenemos 70 unidades finales con rotación fija y 90 cuando la demanda cae a cero. Mostramos así que podemos ocultar faltantes o acumulación al mantener una demanda invariable. No presentamos estas acciones ilustrativas como una ejecución de la política entrenada.

### Relación con los resultados de producción

* Relacionamos los **23 stockouts semanales** con la posibilidad de picos o cambios de demanda ausentes del modelo. Identificamos un mecanismo compatible, sin estimar cuántos incidentes explica ni equiparar unidades no atendidas con stockouts.
* Relacionamos el costo de almacenamiento de **8400 frente a 3200** con posibles acumulaciones cuando cae la rotación. Calculamos una razón de **2.625 veces el objetivo**, sin atribuir toda la diferencia al supuesto de demanda fija.
* Relacionamos los **41 productos vencidos por semana** con la posible permanencia de productos al caer la demanda. Además, observamos que detenemos el contador de vencimiento en 1 y no descontamos del inventario unidades vencidas en la transición.
* Reportamos la satisfacción de **0.61** como una métrica proporcionada. Al no conocer su definición ni contar con registros de servicio, no la interpretamos automáticamente como un porcentaje de clientes satisfechos ni establecemos una relación causal con los faltantes.

Distinguimos otras simplificaciones del estado y la transición: aplicamos la reposición sin demora, no registramos ventas perdidas al recortar el inventario a cero y no distinguimos lotes con diferentes vencimientos. Señalamos estas limitaciones dentro de nuestro componente, sin modificar los de otros grupos.

**Conclusión.** Sustentamos el argumento en la demanda invariable de la transición, en las diferencias de los balances al variar únicamente la demanda y en la compatibilidad de esos mecanismos con los síntomas reportados. Necesitaríamos series diarias de demanda, inventario, pedidos y vencimientos para confirmar su contribución causal. Con la evidencia disponible identificamos una limitación estructural, pero no podemos repartir los incidentes entre causas.


In [6]:
razon_costo = (
    resultados_produccion['costo_almacenamiento_semanal']
    / resultados_produccion['costo_objetivo_semanal']
)
print(f'Costo de almacenamiento / objetivo: {razon_costo:.3f}')
print('Verificamos los ejemplos de los entregables 1.1 y 1.2.')


Costo de almacenamiento / objetivo: 2.625
Verificamos los ejemplos de los entregables 1.1 y 1.2.


## Entregable 1.3: MDP corregido

Partimos de las dos limitaciones que identificamos en los entregables 1.1 y 1.2: el estado no incluye las unidades en tránsito, con lo que dos historias distintas colapsan en el mismo estado observado y producen inventarios siguientes distintos; y la transición congela el nivel de demanda, con lo que el simulador nunca representa estacionalidad ni cambios de tendencia. Proponemos un estado aumentado

$$s_t = (I_t, E_t, L_t, T_t, D_t)$$

donde $I_t$ es el inventario, $E_t$ los días hasta el vencimiento, $L_t$ el nivel de demanda, $T_t$ las unidades en tránsito (pedido del día anterior, aún no recibido) y $D_t$ la tendencia de la demanda (bajando, estable o subiendo). Mantenemos $I_t$, $E_t$ y $L_t$ con el mismo dominio que el MDP original, para que el crecimiento del espacio de estados que calculamos en el entregable 1.4 se explique únicamente por las dos variables nuevas y no por un cambio de granularidad.

### Restauramos la propiedad de Markov con $T_t$

En el entregable 1.1 mostramos que, con estado $(20,14,\text{medio})$ y acción $0$, una entrega pendiente de 30 unidades produce un inventario siguiente distinto a no tener ninguna entrega pendiente, aunque el estado observado es idéntico. Al incluir $T_t$ en el estado, esas dos historias dejan de colapsar: pasan a ser los estados $(20,14,\text{medio},30,\text{estable})$ y $(20,14,\text{medio},0,\text{estable})$, y la transición

$$I_{t+1} = \text{discretizar}\big(\max(0,\ \min(100, I_t+T_t) - d(L_t))\big)$$

produce, para cada uno, un resultado determinista propio. Ya no necesitamos conocer la historia completa para predecir $I_{t+1}$, solo el estado aumentado y la acción, que es exactamente la propiedad que definimos en el entregable 1.1.

No afirmamos que $T_t$ agote toda la información oculta que discutimos en ese entregable: seguimos sin representar lotes con vencimientos distintos ni el historial completo de la demanda de siete días. Es una corrección dirigida al mecanismo concreto que documentamos con el contraejemplo de los pedidos pendientes, no una prueba general de que el estado aumentado sea suficiente.

### Dejamos que la demanda cambie con $D_t$

En el entregable 1.2 señalamos que `new_demand = demand_level` congela la demanda dentro de todo el episodio, y que eso le impide al simulador representar picos ni caídas de temporada. Modelamos $D_t$ como una cadena de Markov sobre tres tendencias (bajando, estable, subiendo) y hacemos que $L_{t+1}$ se muestree según una distribución que depende de $D_t$ y de $L_t$: con tendencia "subiendo" es más probable pasar a niveles de demanda más altos, y con "bajando" más probable bajar. No proponemos estas probabilidades como una estimación calibrada con datos reales de la farmacia, no los tenemos; las usamos como un supuesto ilustrativo, razonable dentro del rango que describe el problema, suficiente para que el simulador deje de tener una demanda estática.

### Devolvemos la demanda no atendida de forma explícita

La línea `new_inventory = max(0, ...)` del código original descarta la magnitud del faltante: un estado con demanda de 40 y disponible de 39 se ve igual que uno con disponible de 0. Modificamos la transición para que retorne, además del estado siguiente, la cantidad de unidades no atendidas en ese paso. Esto no cambia el estado ni la dinámica del inventario, solo expone una cantidad que ya se calculaba internamente y que el Grupo 2 necesita para penalizar el faltante de forma proporcional en vez de binaria.

### Compatibilidad con el resto del sistema

El enunciado pide que la versión corregida sea compatible con el resto del sistema. Los Grupos 3, 4 y 6 reciben estados en el formato original de tres variables. Agregamos dos funciones de proyección, `estado_a_original` y `estado_desde_original`, para que el código de esos grupos pueda seguir operando sobre el formato de tres variables sin reescribirse, inicializando $T_t=0$ y $D_t=\text{estable}$ cuando no hay información adicional disponible.


In [7]:
import numpy as np

DIAS_VENCIMIENTO = [1, 7, 14, 30, 60]
NIVELES_DEMANDA = ['bajo', 'medio', 'alto', 'crítico']
UNIDADES_TRANSITO = [0, 10, 20, 30, 40, 50]
TENDENCIAS_DEMANDA = ['bajando', 'estable', 'subiendo']
DEMAND_MAP = {'bajo': 5, 'medio': 15, 'alto': 25, 'crítico': 40}

# Probabilidad de que la demanda pase a cada nivel, condicionada a la tendencia
# actual y al nivel actual. Es un supuesto ilustrativo, no calibrado con datos
# reales, que hace que la demanda suba con mas probabilidad cuando la tendencia
# es 'subiendo', y baje con mas probabilidad cuando es 'bajando'.
PROB_DEMANDA = {
    'bajando': {
        'bajo':    {'bajo': 0.80, 'medio': 0.15, 'alto': 0.04, 'crítico': 0.01},
        'medio':   {'bajo': 0.40, 'medio': 0.45, 'alto': 0.12, 'crítico': 0.03},
        'alto':    {'bajo': 0.10, 'medio': 0.35, 'alto': 0.45, 'crítico': 0.10},
        'crítico': {'bajo': 0.05, 'medio': 0.20, 'alto': 0.35, 'crítico': 0.40},
    },
    'estable': {
        'bajo':    {'bajo': 0.70, 'medio': 0.20, 'alto': 0.08, 'crítico': 0.02},
        'medio':   {'bajo': 0.15, 'medio': 0.65, 'alto': 0.15, 'crítico': 0.05},
        'alto':    {'bajo': 0.05, 'medio': 0.20, 'alto': 0.60, 'crítico': 0.15},
        'crítico': {'bajo': 0.02, 'medio': 0.08, 'alto': 0.25, 'crítico': 0.65},
    },
    'subiendo': {
        'bajo':    {'bajo': 0.50, 'medio': 0.30, 'alto': 0.15, 'crítico': 0.05},
        'medio':   {'bajo': 0.05, 'medio': 0.45, 'alto': 0.35, 'crítico': 0.15},
        'alto':    {'bajo': 0.02, 'medio': 0.10, 'alto': 0.48, 'crítico': 0.40},
        'crítico': {'bajo': 0.01, 'medio': 0.04, 'alto': 0.20, 'crítico': 0.75},
    },
}

# Probabilidad de que la tendencia misma cambie. Es persistente (alta
# probabilidad de mantenerse) pero puede transicionar de forma gradual.
PROB_TENDENCIA = {
    'bajando':  {'bajando': 0.60, 'estable': 0.35, 'subiendo': 0.05},
    'estable':  {'bajando': 0.20, 'estable': 0.60, 'subiendo': 0.20},
    'subiendo': {'bajando': 0.05, 'estable': 0.35, 'subiendo': 0.60},
}


def discretizar_inventario(inv):
    """Redondea el inventario al multiplo de 10 mas cercano, acotado en [0, 100].

    El propio codigo original ya asume una grilla de multiplos de 10 en su
    comentario de cabecera, pero su funcion transition() nunca redondea a esa
    grilla (con demanda 'bajo'=5 el inventario cae en 55, 45, valores que no
    son multiplos de 10, como vimos en el entregable 1.2). Para poder usar una
    tabla Q finita como la que espera el Grupo 3, discretizamos explicitamente.
    """
    return int(min(100, max(0, round(inv / 10.0)))) * 10


def discretizar_dias(dias):
    """Mapea los dias restantes al valor mas cercano dentro de la grilla declarada."""
    return min(DIAS_VENCIMIENTO, key=lambda x: abs(x - dias))


def transition_corregida(state, action, rng=None):
    """Transicion del MDP corregido.

    state = (inventario, dias_vencimiento, demanda_nivel, unidades_en_transito, tendencia)
    action: unidades pedidas hoy, en {0, 10, 20, 30, 40, 50}
    rng: generador de numpy para muestrear la demanda y la tendencia siguientes;
         si es None usamos el valor mas probable de cada distribucion (version
         determinista, util para depurar y para comparar contra el original).

    Retorna (estado_siguiente, demanda_no_atendida).
    """
    inventario, dias_vencimiento, demanda_nivel, en_transito, tendencia = state

    # El pedido de ayer llega hoy antes de servir la demanda. Este es el
    # cambio que restaura la propiedad de Markov (entregable 1.1).
    inventario_efectivo = min(100, inventario + en_transito)

    demanda_hoy = DEMAND_MAP[demanda_nivel]
    demanda_no_atendida = max(0, demanda_hoy - inventario_efectivo)
    new_inventory = discretizar_inventario(max(0, inventario_efectivo - demanda_hoy))
    new_days = discretizar_dias(max(1, dias_vencimiento - 1))

    probs_d = PROB_DEMANDA[tendencia][demanda_nivel]
    probs_t = PROB_TENDENCIA[tendencia]
    if rng is None:
        new_demand = max(probs_d, key=probs_d.get)
        new_tendencia = max(probs_t, key=probs_t.get)
    else:
        # str(...) porque rng.choice sobre una lista de str devuelve un
        # escalar numpy (np.str_), y preferimos mantener el estado como
        # tipos nativos de Python en toda la tupla.
        new_demand = str(rng.choice(list(probs_d.keys()), p=list(probs_d.values())))
        new_tendencia = str(rng.choice(list(probs_t.keys()), p=list(probs_t.values())))

    # El pedido de hoy queda en transito para manana.
    new_transito = action

    next_state = (new_inventory, new_days, new_demand, new_transito, new_tendencia)
    return next_state, int(demanda_no_atendida)


def estado_a_original(state):
    """Proyecta el estado corregido (5 variables) al formato original (3 variables),
    para que el codigo de los Grupos 3, 4 y 6 siga funcionando sin cambios."""
    inventario, dias_vencimiento, demanda_nivel, _, _ = state
    return (inventario, dias_vencimiento, demanda_nivel)


def estado_desde_original(state, en_transito=0, tendencia='estable'):
    """Expande un estado original (3 variables) al formato corregido (5 variables),
    con valores neutros por defecto cuando no hay informacion adicional."""
    inventario, dias_vencimiento, demanda_nivel = state
    return (inventario, dias_vencimiento, demanda_nivel, en_transito, tendencia)


### Verificamos que el estado aumentado restaura Markov

Repetimos el contraejemplo del entregable 1.1, ahora con el estado aumentado en lugar del original de tres variables.


In [8]:
estado_con_pedido = (20, 14, 'medio', 30, 'estable')
estado_sin_pedido = (20, 14, 'medio', 0, 'estable')

siguiente_con_pedido, faltante_con_pedido = transition_corregida(estado_con_pedido, 0)
siguiente_sin_pedido, faltante_sin_pedido = transition_corregida(estado_sin_pedido, 0)

print('Estado aumentado con entrega pendiente:', estado_con_pedido)
print('Inventario siguiente:', siguiente_con_pedido[0])
print()
print('Estado aumentado sin entrega pendiente:', estado_sin_pedido)
print('Inventario siguiente:', siguiente_sin_pedido[0])
print()
print('En el entregable 1.1, ambas historias compartian el mismo estado observado (20, 14, medio)')
print('y producian inventarios distintos (35 y 5) sin que el estado por si solo lo explicara.')
print('Ahora cada estado aumentado predice su propio resultado de forma determinista.')

repeticiones = [transition_corregida(estado_con_pedido, 0)[0][0] for _ in range(5)]
assert len(set(repeticiones)) == 1
assert siguiente_con_pedido[0] != siguiente_sin_pedido[0]


Estado aumentado con entrega pendiente: (20, 14, 'medio', 30, 'estable')
Inventario siguiente: 40

Estado aumentado sin entrega pendiente: (20, 14, 'medio', 0, 'estable')
Inventario siguiente: 0

En el entregable 1.1, ambas historias compartian el mismo estado observado (20, 14, medio)
y producian inventarios distintos (35 y 5) sin que el estado por si solo lo explicara.
Ahora cada estado aumentado predice su propio resultado de forma determinista.


Los valores concretos que obtenemos ahora (40 y 0) no son los mismos 35 y 5 que calculamos en el entregable 1.1, porque ahí usábamos la aritmética exacta sin discretizar. Aquí redondeamos al múltiplo de 10 más cercano para poder usar una tabla Q finita, y 35 queda en 40 por la convención de redondeo de Python al par más cercano. El punto que queríamos demostrar se mantiene: las dos historias, que antes competían por la misma entrada de la tabla Q, ahora ocupan estados distintos y cada uno tiene su propia predicción determinista.

### Verificamos que la demanda ya no está congelada

En el entregable 1.2 documentamos que la transición original nunca cambia `demand_level`: la trayectoria de 4 días que construimos ahí se queda en "bajo" todo el tiempo. Simulamos 30 pasos consecutivos de la transición corregida partiendo de demanda baja con tendencia subiendo, para confirmar que el nivel de demanda ahora se mueve.


In [9]:
rng = np.random.default_rng(0)
estado = (50, 30, 'bajo', 0, 'subiendo')
niveles_observados = []
for _ in range(30):
    estado, _ = transition_corregida(estado, 20, rng=rng)
    niveles_observados.append(estado[2])

print('Niveles de demanda en 30 pasos consecutivos, partiendo de bajo con tendencia subiendo:')
print(niveles_observados)
print()
print('Niveles distintos observados:', sorted(set(niveles_observados)))
assert len(set(niveles_observados)) > 1


Niveles de demanda en 30 pasos consecutivos, partiendo de bajo con tendencia subiendo:


['medio', 'bajo', 'medio', 'medio', 'medio', 'alto', 'alto', 'alto', 'alto', 'medio', 'bajo', 'bajo', 'bajo', 'crítico', 'crítico', 'crítico', 'alto', 'alto', 'alto', 'crítico', 'crítico', 'crítico', 'crítico', 'alto', 'medio', 'alto', 'crítico', 'alto', 'alto', 'medio']

Niveles distintos observados: ['alto', 'bajo', 'crítico', 'medio']


A diferencia del original, donde `demand_level` se mantiene igual durante todo el episodio, aquí la demanda recorre varios niveles dentro de los mismos 30 pasos porque la tendencia "subiendo" empuja la distribución hacia niveles más altos.

### Verificamos la compatibilidad con el formato original

Confirmamos que un estado corregido se puede proyectar al formato de tres variables que usan los Grupos 3, 4 y 6, y que un estado en ese formato original se puede expandir de vuelta con valores neutros.


In [10]:
estado_corregido_ejemplo = (40, 14, 'alto', 20, 'estable')
proyectado = estado_a_original(estado_corregido_ejemplo)
expandido = estado_desde_original((40, 14, 'alto'))

print('Estado corregido:', estado_corregido_ejemplo)
print('Proyectado al formato original de tres variables:', proyectado)
print()
print('Estado original recibido de otro grupo:', (40, 14, 'alto'))
print('Expandido al formato corregido con valores neutros por defecto:', expandido)

assert proyectado == (40, 14, 'alto')
assert expandido == (40, 14, 'alto', 0, 'estable')


Estado corregido: (40, 14, 'alto', 20, 'estable')
Proyectado al formato original de tres variables: (40, 14, 'alto')

Estado original recibido de otro grupo: (40, 14, 'alto')
Expandido al formato corregido con valores neutros por defecto: (40, 14, 'alto', 0, 'estable')


Estas funciones permiten adaptar el formato de los estados entre tres y cinco variables. La proyección descarta tránsito y tendencia, por lo que su uso en la representación del agente debe evaluarse junto con la información que necesita para decidir. La Pregunta 4 desarrolla esta distinción y las interfaces requeridas por las entregas recibidas.

## Entregable 1.4: impacto sobre el espacio de estados

Mantenemos la misma granularidad para inventario, días y demanda que el MDP original, y solo agregamos las dos variables nuevas, para que el crecimiento del espacio de estados se explique exclusivamente por unidades en tránsito y tendencia de demanda, no por un cambio de resolución en las variables existentes.

Antes de construir la tabla verificamos un detalle del código original: el comentario de cabecera declara 10 niveles de inventario (múltiplos de 10 entre 0 y 100), pero esa misma lista tiene en realidad 11 valores, y ya mostramos en el entregable 1.2 que la función `transition()` ni siquiera discretiza el inventario a esa grilla (con demanda "bajo" obtuvimos 55 y 45, que no son múltiplos de 10). Presentamos la tabla usando el número que declara el comentario, 10 niveles y 200 estados, porque es la cifra que el enunciado y el Grupo 7 ya usan como referencia compartida en el entregable 7.2. Verificamos abajo que el factor de expansión no cambia si en cambio usamos el conteo literal de 11 niveles: la ambigüedad de la granularidad del inventario se cancela entre el numerador y el denominador.


In [11]:
niveles_inventario_declarados = 10
niveles_inventario_reales = len(list(range(0, 101, 10)))
print('Niveles de inventario que declara el comentario original:', niveles_inventario_declarados)
print('Niveles de inventario que enumera realmente ese mismo comentario:', niveles_inventario_reales)
print()

acciones = 6
dias = len(DIAS_VENCIMIENTO)
demandas = len(NIVELES_DEMANDA)
transitos = len(UNIDADES_TRANSITO)
tendencias = len(TENDENCIAS_DEMANDA)

total_original = niveles_inventario_declarados * dias * demandas
total_corregido = niveles_inventario_declarados * dias * demandas * transitos * tendencias
factor = total_corregido / total_original

total_original_11 = niveles_inventario_reales * dias * demandas
total_corregido_11 = niveles_inventario_reales * dias * demandas * transitos * tendencias
factor_11 = total_corregido_11 / total_original_11

print(f'{"Dimension":38} {"Original":>10} {"Corregido":>12}')
print(f'{"Niveles de inventario":38} {niveles_inventario_declarados:>10} {niveles_inventario_declarados:>12}')
print(f'{"Dias hasta vencimiento":38} {dias:>10} {dias:>12}')
print(f'{"Niveles de demanda":38} {demandas:>10} {demandas:>12}')
print(f'{"Unidades en transito (nuevo)":38} {"---":>10} {transitos:>12}')
print(f'{"Tendencia de demanda (nuevo)":38} {"---":>10} {tendencias:>12}')
print()
print(f'{"Total de estados":38} {total_original:>10} {total_corregido:>12}')
print(f'{"Pares (estado, accion)":38} {total_original*acciones:>10} {total_corregido*acciones:>12}')
print(f'{"Entradas de la tabla Q":38} {total_original*acciones:>10} {total_corregido*acciones:>12}')
print()
print('Factor de expansion usando 10 niveles de inventario:', factor)
print('Factor de expansion usando 11 niveles de inventario:', factor_11)


Niveles de inventario que declara el comentario original: 10
Niveles de inventario que enumera realmente ese mismo comentario: 11

Dimension                                Original    Corregido
Niveles de inventario                          10           10
Dias hasta vencimiento                          5            5
Niveles de demanda                              4            4
Unidades en transito (nuevo)                  ---            6
Tendencia de demanda (nuevo)                  ---            3

Total de estados                              200         3600
Pares (estado, accion)                       1200        21600
Entradas de la tabla Q                       1200        21600

Factor de expansion usando 10 niveles de inventario: 18.0
Factor de expansion usando 11 niveles de inventario: 18.0


### Qué implica esto para el algoritmo de aprendizaje

Con 18 veces más estados, cualquier método tabular como el que implementa el Grupo 3 necesita visitar 18 veces más pares (estado, acción) para alcanzar la misma cobertura promedio que tenía con el MDP original. El código compartido no incluye la implementación de `env`, así que no conocemos la duración real de un episodio; usamos un valor ilustrativo de 30 pasos por episodio solo para dimensionar el orden de magnitud, no como una cifra exacta.


In [12]:
episodios_originales = 1000
pasos_por_episodio_supuesto = 30  # supuesto ilustrativo, el env no esta disponible

visitas_promedio_original = (episodios_originales * pasos_por_episodio_supuesto) / (total_original * acciones)
episodios_necesarios_corregido = visitas_promedio_original * (total_corregido * acciones) / pasos_por_episodio_supuesto

print('Visitas promedio por par (estado, accion) en el original con 1000 episodios:', round(visitas_promedio_original, 2))
print('Episodios necesarios en el corregido para mantener esa misma cobertura promedio:', round(episodios_necesarios_corregido))


Visitas promedio por par (estado, accion) en el original con 1000 episodios: 25.0
Episodios necesarios en el corregido para mantener esa misma cobertura promedio: 18000


Necesitaríamos del orden de 18000 episodios en vez de 1000 para mantener la misma cobertura promedio por par, solo por el crecimiento del espacio de estados, sin considerar todavía si $\varepsilon=0.05$ alcanza para visitar los estados de demanda crítica, que corresponde al análisis del Grupo 5. Esto también se conecta con el esquema de decaimiento de $\alpha$ que le corresponde justificar al Grupo 3 en el entregable 3.2: con más estados y menos visitas por estado en las primeras iteraciones, un $\alpha$ constante y alto como 0.9 tarda más en estabilizarse, porque cada actualización individual pesa más sobre una entrada de la tabla que se visita con menos frecuencia.

Los resultados recibidos de G3 y G5 permiten contrastar esta hipótesis en las preguntas de integración. Sus coberturas corresponden a entornos y horizontes distintos, que identificamos al relacionar sus métricas con nuestro espacio ampliado.


## Prompts de apoyo

### Entregables 1.1 y 1.2

> Queremos que nos ayudes a entender los entregables 1.1 y 1.2. Analiza el código Python y explícanos qué contiene para posteriormente completar estos entregables. No queremos que nos resuelvas todo; queremos que nos ayudes a plantear las fórmulas y los diferentes cambios que tendríamos que hacer para que todo funcione correctamente.

**Propósito del prompt:** buscamos comprender la representación del estado y la función de transición antes de completar los entregables. Delimitamos el apoyo al planteamiento de fórmulas y al análisis de los cambios necesarios para poder justificar nuestras decisiones.

### Entregables 1.3 y 1.4

> Ya tenemos un avance de un integrante del equipo con una versión corregida del MDP y una tabla de impacto en un archivo aparte. Ayúdanos a completar los entregables 1.3 y 1.4 del examen en el notebook del grupo, basándote en ese avance y en lo que ya respondimos en los entregables 1.1 y 1.2, verificando con código que la propiedad de Markov realmente se restaura y que los números de la tabla son correctos.

**Propósito del prompt:** buscamos integrar un avance ya existente del equipo con el resto del notebook, manteniendo la misma línea argumentativa de los entregables anteriores y verificando con ejecuciones concretas, no solo con texto, que las correcciones propuestas efectivamente resuelven los problemas que documentamos en 1.1 y 1.2.

**Por qué funcionó:** el prompt señala explícitamente los archivos de partida, el avance del compañero y el notebook con 1.1 y 1.2 ya resueltos, en vez de pedir una implementación genérica de un MDP corregido, y pide verificación ejecutable en vez de solo una justificación en prosa, que es el mismo estándar que ya veníamos usando en el resto del notebook.

### Integración y reflexión grupal

**Prompt utilizado, síntesis de la solicitud:** Completa las cuatro respuestas de integración y la reflexión en el notebook de Grupo1, revisando las entregas de los siete grupos. Analiza las conexiones entre las implementaciones actuales y sus resultados, con énfasis en pensamiento sistémico e identificación de errores fatales. Usa párrafos de máximo cuatro oraciones, sin lítotes ni ironía, y conserva los códigos de los otros grupos.

**Propósito y utilidad:** La solicitud orientó la revisión hacia dependencias compartidas: significado de la acción, disponibilidad de inventario, estado observable y criterio de evaluación. Las respuestas distinguen resultados medidos, insumos citados por terceros y metas propuestas para la integración. Ese enfoque permite justificar decisiones del sistema con evidencia de cada componente.


## Referencias de apoyo

- GeeksforGeeks. (2025a, July 23). How to calculate moving averages in Python? GeeksforGeeks. https://www.geeksforgeeks.org/python/how-to-calculate-moving-averages-in-python/ 
- GeeksforGeeks. (2025b, July 31). Markov Chain. GeeksforGeeks. https://www.geeksforgeeks.org/machine-learning/markov-chain/ 
- GeeksforGeeks. (2026, July 8). Markov decision process. GeeksforGeeks. https://www.geeksforgeeks.org/machine-learning/markov-decision-process/
- Puterman, M. L. (1994). *Markov Decision Processes: Discrete Stochastic Dynamic Programming*. John Wiley & Sons. Base teórica de la aumentación de estado que usamos en el entregable 1.3 para restaurar la propiedad de Markov.


## Integración de resultados

Leemos las implementaciones como decisiones sobre un sistema compartido: el MDP determina qué ocurre, la recompensa valora el resultado, el aprendizaje y la exploración construyen la política, y la evaluación comprueba sus consecuencias operativas. Usamos los archivos y las salidas disponibles, con una reproducción puntual de la comparación de G5. Los resultados de experimentos distintos conservan su configuración y procedencia al citarlos.

| Grupo | Fuente consultada | Alcance utilizado |
|---|---|---|
| 1 | Este notebook y [mdp_corregido.py](mdp_corregido.py) | Estado, transición y tabla de impacto |
| 2 | [Grupo_2_Para_Integracion.ipynb](../Grupo2/Grupo_2_Para_Integracion.ipynb) | Cinco componentes de recompensa y comparación de políticas |
| 3 | [algoritmo_corregido.py](../Grupo3/src/algoritmo_corregido.py), [resumen_ablacion.json](../Grupo3/resultados/resumen_ablacion.json) y [triada_mortal.py](../Grupo3/src/triada_mortal.py) | Aprendizaje configurable, métricas de ablación y experimento lineal propio |
| 4 | Carpeta Grupo4 y apartado de compatibilidad de [grupo7.ipynb](../Grupo7/grupo7.ipynb) | Propuesta de 12 características comunicada por G7; código y MSE pendientes de verificación directa |
| 5 | [parcial (1).py](<../Grupo5/parcial (1).py>) | Comparación reproducida de epsilon-greedy y Optimista+UCB |
| 6 | [Verificación empírica](<../Grupo6/S10 - Verificacion Empirica Reward Hacking.ipynb>) | Diagnóstico y curvas guardadas del entorno reconstruido |
| 7 | [grupo7.ipynb](../Grupo7/grupo7.ipynb) y [protocolo_evaluacion.py](../Grupo7/protocolo_evaluacion.py) | Gap, integración reportada y criterios de evaluación |


# Pregunta 1: Diagnóstico sistémico

**Usando el MDP corregido del Grupo 1, la función de recompensa corregida del Grupo 2, y la proyección de curvas del Grupo 6: si se aplican únicamente las correcciones del MDP y la función de recompensa sin cambiar el algoritmo ni la exploración, ¿predicen que el agente aprenderá una política mejor? Justifiquen usando las métricas concretas que cada grupo produjo.**

Predecimos una mejora parcial y condicionada: el MDP ampliado permite distinguir entregas pendientes y cambios de demanda, y la recompensa corregida valora el servicio y los costos de esas decisiones. El aprendizaje conserva un presupuesto limitado y una exploración de 0.05, de modo que la nueva información puede quedar representada en estados con poca experiencia. La mejora efectiva depende de que transición, recompensa y agente interpreten el mismo proceso de inventario.

Nuestro cambio aumenta el espacio nominal de 200 a 3600 estados y de 1200 a 21 600 pares estado–acción. Con el mismo presupuesto de 1000 episodios de 30 pasos, la experiencia disponible por par disminuye de 25 a aproximadamente 1.39 actualizaciones teóricas. Por tanto, mejorar la representación del entorno puede reducir la cobertura si G3 y G5 mantienen el mismo esquema de aprendizaje y exploración.


El Grupo 2 muestra que la señal corregida favorece una política que ajusta el pedido al estado. Su comparación usa 1000 episodios de 30 pasos, entrega inmediata y una política que selecciona directamente la mejor recompensa inmediata en cada paso. Por ello, sus resultados cuantifican el cambio de incentivos y el desempeño de dos políticas predefinidas.

| Métrica del Grupo 2 | Siempre pedir 50 | Política de recompensa inmediata corregida |
|---|---:|---:|
| Recompensa acumulada promedio | −842.29 | 144.40 |
| Inventario promedio | 97.98 | 5.76 |
| Unidades pedidas por episodio | 1500 | 582.94 |
| Pasos con inventario en ventana de vencimiento | 18.11 | 7.483 |
| Pasos con faltante por episodio | 0 | 0 |

La integración G1–G2 también exige que ambos componentes interpreten igual el momento de llegada del pedido. En G1, un pedido realizado hoy llega en el siguiente paso. Si G2 lo considera disponible inmediatamente, la recompensa puede premiar servicio que todavía no ocurrió. La recompensa debe calcularse con el inventario realmente disponible y conservar el costo del pedido realizado.

Los resultados de G6 apoyan que las correcciones cambian el aprendizaje, pero no prueban que G1 y G2 por sí solos produzcan una política mejor. Su línea base reporta TD de 51.33, mientras que la combinación G1+G2+G3+G5 reporta 23.14. Esto indica que la mejora depende de la interacción entre representación, recompensa, algoritmo y exploración.

Esperamos que, al conciliar las interfaces, disminuya el incentivo a acumular inventario y aumente la sensibilidad a entregas y demanda. También esperamos mayor variabilidad inicial por la demanda estocástica y por las visitas escasas a algunos pares del nuevo estado. Una política estable y mejor en servicio sigue siendo una hipótesis por verificar con las implementaciones reales y el presupuesto original. Esta predicción utiliza las métricas de G1, G2 y G6, y recoge la extensión de G7 con su alcance experimental explícito.


# Pregunta 2: Causa raíz
Usando el algoritmo corregido del Grupo 3, las métricas de cobertura del Grupo 5, y el análisis de gap del Grupo 7: identifiquen el problema raíz que más contribuye al gap entre simulación y producción. ¿Es el error de implementación del algoritmo, la cobertura insuficiente del espacio de estados, o el reward hacking? Apoyen su respuesta con evidencia cuantitativa de los tres grupos.

Para el gap global priorizamos el reward hacking como causa raíz del objetivo: el sistema recompensa acumular inventario, y su evaluación reutiliza ese criterio para decidir si funciona bien. Para el síntoma específico de stockouts, la evidencia de G3, G5 y G7 señala la cobertura insuficiente y la decisión por defecto como el mecanismo de aprendizaje más directo. Ambos procesos pueden coexistir: unas situaciones reciben decisiones que acumulan stock y otras reciben decisiones basadas en experiencia escasa. La prioridad expresa nuestro diagnóstico técnico; los agregados disponibles dejan pendiente cuantificar la contribución causal de cada componente en producción.

G3 muestra esta dependencia entre algoritmo y exploración. El valor capturado aumenta de 30.63 % a 73.11 % al mejorar la exploración, mientras la corrección completa alcanza 75.88 %. El cambio aislado a SARSA solo alcanza 33.81 %. Esto indica que corregir el algoritmo sin cambiar cómo se recopila experiencia tiene un efecto limitado.

| Configuración de G3 | Valor capturado | Cobertura de pares |
|---|---:|---:|
| Original | 30.63 % | 5.23 % |
| Desempate aleatorio | 68.46 % | 4.58 % |
| Mayor exploración inicial con decaimiento | 73.11 % | 8.36 % |
| Cambio aislado a SARSA | 33.81 % | 5.38 % |
| Todas sus correcciones | 75.88 % | 8.21 % |

La mayor ganancia aislada de esa tabla corresponde al cambio de exploración, y el desempate también modifica considerablemente el valor capturado. Incluso una cobertura total menor puede acompañar mejores decisiones cuando cambia la experiencia recogida en los estados relevantes. La expresión original `Q[next_state][argmax(Q[next_state])]` implementa el máximo de Q-learning; adoptar SARSA cambia la política que se evalúa y exige usar la siguiente acción de comportamiento. Esto sitúa la discusión del algoritmo en su relación con exploración, distribución de visitas y recompensa.

G5 llega al mismo problema desde la cobertura. Epsilon-greedy visita 64.2 % de los pares estado–acción, mientras Optimista+UCB alcanza 99.6 %. Los pares críticos pendientes disminuyen de 150 a 5. Una política puede visitar todos los estados y aun así conocer pocas acciones dentro de ellos.


G7 conecta estos problemas con producción: 23 stockouts, 41 productos vencidos por semana, costo de 8400 frente a 3200 y satisfacción de 0.61. Además, Optimista+UCB elimina faltantes en su prueba de estrés, pero mantiene inventario promedio de 80.69 bajo la recompensa original. Mejorar la cobertura puede reducir stockouts sin corregir el incentivo a sobreabastecer. Por eso, el problema sistémico combina una recompensa mal alineada con experiencia insuficiente en partes del espacio.


Esta lectura explica nuestra prioridad,  corregir el objetivo atiende el sobreabastecimiento que un agente mejor entrenado también podría aprender, y mejorar la cobertura atiende las decisiones poco informadas que generan faltantes. El protocolo de G7 conecta ambas intervenciones y permite detectar si una mejora local desplaza el problema hacia otra métrica. La asignación de porcentajes de responsabilidad requiere registros de demanda, acciones, llegadas y vencimientos de cada incidente.


# Pregunta 3: Plan de corrección mínimo viable
Usando los resultados de todos los grupos: diseñen el plan de corrección mínimo viable que resolvería el gap de producción. Para cada cambio especifiquen qué componente modifica, qué métrica mejora, y cuánto esperan que mejore. El plan debe ser implementable en dos semanas de trabajo e indicar el orden de prioridad de cada corrección


El plan debe corregir primero las interfaces que conectan los componentes. Después debe modificar recompensa y exploración. Finalmente debe evaluar el sistema completo con una misma dinámica y las mismas métricas.

| Prioridad | Días | Componente e integración | Cambio | Métrica objetivo |
|---|---:|---|---|---|
| 1 | 1–2 | G1 → G2/G3 | Unificar orden de demanda, llegada y pedido. Corregir el contador de vencimiento y conservar el estado completo. | Cero diferencias en servicio y faltantes entre transición y recompensa. |
| 2 | 3–4 | G2 → G1 | Calcular recompensa con inventario realmente disponible y faltantes producidos por G1. | Reducir inventario y pedidos sin aumentar faltantes. Referencia G2: retorno −842.29 → 144.40 e inventario 97.98 → 5.76. |
| 3 | 5–7 | G3 ↔ G5 | Comparar el entrenamiento corregido con una estrategia de exploración de mayor cobertura. | Referencias: valor capturado 30.63 % → 75.88 % en G3 y cobertura 64.2 % → 99.6 % en G5. |
| 4 | 8–9 | G4 → G3 | Si se usa aproximación, conservar tránsito y tendencia en las características y normalizar sus escalas. | Cero divergencias en cinco semillas. |
| 5 | 10–12 | G6 → todos | Ejecutar línea base, G1+G2 y sistema integrado bajo el mismo entorno, semillas y horizonte. | Mejor servicio, menos faltantes, menor inventario y menor costo frente a la línea base. |
| 6 | 13–14 | G7 → todos | Aplicar criterios de aceptación y prueba de estrés antes del piloto. | Metas provisionales: stockouts ≤11/semana, vencidos ≤20/semana y costo ≤3840. |

El orden importa porque cada componente consume la salida del anterior. No tiene sentido ajustar exploración si transición y recompensa describen procesos distintos. Tampoco conviene aprobar una política por retorno o TD si los indicadores de producción siguen deteriorados.

Las mejoras reportadas por cada grupo no deben sumarse como si fueran independientes. Cada resultado fue obtenido bajo configuraciones distintas. El paquete final debe evaluarse en un entorno común para medir qué mejoras sobreviven a la integración.

# Pregunta 4: Compatibilidad de correcciones
El Grupo 1 corrigió el MDP, el Grupo 4 mejoró el preprocesamiento, y el Grupo 3 corrigió el algoritmo. ¿Son compatibles estas tres correcciones entre sí? Identifiquen si alguna corrección de un grupo invalida o requiere modificación de la corrección de otro grupo, y propongan cómo resolver cualquier incompatibilidad.


Las correcciones de G1, G3 y G4 son conceptualmente compatibles, pero no pueden conectarse directamente sin adaptar sus interfaces. G1 modifica la información del estado, G4 transforma esa información y G3 la utiliza para aprender. Si una etapa elimina información necesaria, la corrección anterior pierde su efecto.

| Conexión | Problema de integración | Corrección |
|---|---|---|
| G1 → G3 | G1 usa cinco variables y G3 consume estados mediante la interfaz del entorno. | Mantener las cinco variables en `reset` y `step`. |
| G3 → G1 | G3 selecciona índices 0–5 y G1 interpreta unidades pedidas. | Traducir con `[0,10,20,30,40,50][indice]`. |
| G1 → G4 | La propuesta reportada de G4 usa tres variables y omite tránsito y tendencia. | Extender las características para conservar ambas variables. |
| G4 → G3 | La ruta tabular y la aproximación lineal utilizan representaciones distintas. | Elegir una ruta explícita y ajustar pesos y `alpha` a la representación usada. |

El principal riesgo está entre G1 y G4. Si G1 distingue estados por tránsito y tendencia, pero G4 elimina esas variables, estados con entregas futuras diferentes vuelven a verse iguales para G3. En ese caso, el preprocesamiento anula parte de la corrección del MDP.

También existen incompatibilidades de interfaz. G1 devuelve `(estado_siguiente, faltante)`, mientras G3 espera una interfaz tipo `(estado_siguiente, recompensa, done, info)`. Además, confundir un índice de acción con unidades convierte un pedido de 50 en uno de 5. Estas diferencias deben resolverse en el entorno adaptador.

La integración debe conservar el estado completo desde G1, transformar esa información sin perder variables relevantes en G4 y entregar a G3 una interfaz única. Así, cada corrección mantiene su propósito cuando pasa al siguiente componente.

## Reflexión grupal

Al comenzar la auditoría, concentramos el diagnóstico en la información que el estado conservaba y en la demanda que permanecía fija durante cada episodio. Incorporar tránsito y tendencia nos permitió distinguir situaciones que antes compartían una misma tupla. Al revisar las entregas de los demás grupos, comprendimos que esa ampliación también modifica el significado de la recompensa, la experiencia que requiere el agente y la forma de evaluar sus decisiones.

El Grupo 2 nos ayudó a conectar la representación del estado con los incentivos del sistema, pues una recompensa orientada al servicio necesita saber qué unidades están disponibles en el momento de atender la demanda. Nuestro pedido diferido hizo visible esa dependencia, ya que recibir mañana y atender hoy requieren balances distintos. Reconocimos también que el contador de vencimiento de nuestra propuesta se congela al redondear, lo cual afecta la señal que reciben los demás componentes. Estos hallazgos nos llevaron a entender la compatibilidad entre componentes como una responsabilidad compartida sobre el significado de los datos.

Los Grupos 3 y 5 ampliaron nuestra interpretación de la cobertura, pues mostraron que visitar todos los estados puede coexistir con conocer pocas acciones en cada uno. Sus resultados nos llevaron a relacionar el tamaño del espacio de estados con el presupuesto de exploración, el mecanismo de desempate y la distribución de visitas. El aporte atribuido al Grupo 4 por el Grupo 7 añadió la posibilidad de generalizar entre estados, junto con la necesidad de conservar tránsito y tendencia entre las características. Aprendimos así a distinguir qué información conocimos de forma directa y cuál llegó a través de otro equipo.

Las curvas del Grupo 6 y el protocolo del Grupo 7 conectaron este aprendizaje con el criterio de éxito del sistema. La recompensa, el error de diferencia temporal y la frecuencia de una acción describen aspectos que deben interpretarse junto con los faltantes de inventario y los costos asociados. Nuestra conclusión grupal es que cada mejora adquiere sentido al identificar qué información recibe de los demás componentes y qué consecuencias produce en las etapas posteriores del sistema. Por ello proponemos una integración gradual, evidencia comparable entre componentes y una evaluación operativa común para determinar cuándo el sistema está preparado para su uso en producción.


# Repositorio 

https://github.com/SergioAle210/RL-Parcial 